# Nemotron Super v3 Text2SQL LoRA Fine-Tuning using Megatron Bridge

This notebook walks you through dataprep, model training and checkpoint export steps demonstrating how you can fine-tune the Nemotron model for the Text2SQL use-case using LoRA.

## Prerequisites

- You will need a node with 8 x H100 80GB GPU or similar to follow along.
- You will need at least 600GB disk space for checkpoints.

## Launching the container

Launch the NeMo container with this notebook directory mounted. Also note that the command below assumes that your HuggingFace token is stored in the environment variable `$HF_TOKEN`, and that variable is exposed to the container.

```bash
CONTAINER="nvcr.io/nvidian/nemo:26.02.super.rc3"

docker container run --gpus all -it --rm \
  -e HF_TOKEN -e HF_HOME=/root/.cache/huggingface \
  -v $HOME/.cache:/root/.cache \
  -v "$(pwd)":/workspace/notebook \
  --shm-size=16g --net=host --ipc=host \
  --ulimit memlock=-1 --ulimit stack=67108864 \
  "${CONTAINER}" bash
```

Run all other cells below from inside the container.

---
## Configuration

Edit the values below to match your environment.

In [ ]:
import os

n_devices = 8  # How many devices to use for training?
max_seq_len = 4096  # Maximum sequence length for training (in tokens).
hf_model = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16"  # Base model for LoRA training
base_model_path = "/workspace/storage/super_v3/base"  # The path where the converted base model checkpoints are stored.
experiment_name = "baseline" # Human-readable label for this run
dataprep_output_dir = os.path.abspath("results/dataprep")  # Dataprep writes here, training reads from here
train_output_dir = os.path.abspath("results/trained")  # Training checkpoints and logs go here

# Propagate to env so dataprep.py can read them
%env MODEL_ID=$hf_model
%env MAX_SEQ_LEN=$max_seq_len
%env DATAPREP_OUTPUT_DIR=$dataprep_output_dir

---
## Step 1: Data Preparation

Run the data-prep script to build `results/dataprep/training.jsonl`.

In [ ]:
! python dataprep.py

### Sanity Check

Verify the training data was created and set environment variables for the remaining steps.

In [ ]:
training_jsonl = os.path.join(dataprep_output_dir, "training.jsonl")
assert os.path.exists(training_jsonl), f"Expected training data at '{training_jsonl}'. Run the data prep cell first."

with open(training_jsonl) as f:
    n_examples = sum(1 for _ in f)
print(f"Training examples: {n_examples}")
print(f"Dataset: {training_jsonl}")
print(f"Training output dir: {train_output_dir}")

# Set environment variables for subsequent %%bash cells (model conversion + training)
%env N_DEVICES=$n_devices
%env MAX_SEQ_LEN=$max_seq_len
%env HF_MODEL=$hf_model
%env DATASET_DIR=$dataprep_output_dir
%env BASE_MODEL_PATH=$base_model_path
%env EXPERIMENT_NAME=$experiment_name
%env TRAINING_OUTPUT_DIR=$train_output_dir

---
## Step 2: Model Conversion

Convert the HuggingFace checkpoint to Megatron-Bridge format before we can start training.

In [ ]:
%%bash
set -euo pipefail
mkdir -p $BASE_MODEL_PATH

torchrun --nproc-per-node=$N_DEVICES \
  /opt/Megatron-Bridge/examples/conversion/convert_checkpoints.py import \
  --hf-model $HF_MODEL \
  --megatron-path $BASE_MODEL_PATH \
  --tp 1 --ep $N_DEVICES --device-map auto

---
## Step 3: Training

Launch LoRA fine-tuning via `torchrun`. Feel free to inspect `train.py` and modify the training hyperparameters as needed.

In [ ]:
%%bash
set -euo pipefail

# Fresh timestamp each run so repeated executions get unique folders
TIMESTAMP=$(TZ="PST8PDT" date +"%y%m%d-%H%M%S")
export EXPERIMENT_NAME="${TIMESTAMP}-${EXPERIMENT_NAME}"

echo "=== Training Environment ==="
echo "  EXPERIMENT_NAME=$EXPERIMENT_NAME"
echo "  N_DEVICES=$N_DEVICES"
echo "  BASE_MODEL_PATH=$BASE_MODEL_PATH"
echo "  DATASET_DIR=$DATASET_DIR"
echo "  TRAINING_OUTPUT_DIR=$TRAINING_OUTPUT_DIR"
echo "  MAX_SEQ_LEN=$MAX_SEQ_LEN"
echo ""

torchrun --nproc-per-node=$N_DEVICES train.py

---
## Step 4: Merge LoRA Adapter into the Base Model

Merge the LoRA adapter back into the base model and export a single merged checkpoint.
The cell below automatically finds the latest training checkpoint and merges it.

In [ ]:
# Follow the "latest" symlink written by train.py to find the most recent run
latest_run = os.path.join(train_output_dir, "latest")
assert os.path.islink(latest_run), f"No 'latest' symlink found in '{train_output_dir}'. Run the training step first."

checkpoints_dir = os.path.join(os.path.realpath(latest_run), "default", "checkpoints")
assert os.path.isdir(checkpoints_dir), f"Checkpoints dir not found: '{checkpoints_dir}'"

# Read the iteration number written by Megatron-Bridge
iter_file = os.path.join(checkpoints_dir, "latest_checkpointed_iteration.txt")
assert os.path.exists(iter_file), f"'{iter_file}' not found — training may not have saved a checkpoint."
with open(iter_file) as f:
    latest_iter = int(f.read().strip())

lora_checkpoint = os.path.join(checkpoints_dir, f"iter_{latest_iter:07d}")
assert os.path.isdir(lora_checkpoint), f"Checkpoint dir not found: '{lora_checkpoint}'"

# Derive merge output from the selected checkpoint path
merge_output_dir = lora_checkpoint + "_merged_to_base"
print(f"LoRA checkpoint: {lora_checkpoint}")
print(f"Merge output:    {merge_output_dir}")

%env LORA_CHECKPOINT=$lora_checkpoint
%env MERGE_OUTPUT_DIR=$merge_output_dir

In [ ]:
%%bash
set -euo pipefail

echo "=== Merge Environment ==="
echo "  LORA_CHECKPOINT=$LORA_CHECKPOINT"
echo "  BASE_MODEL_PATH=$BASE_MODEL_PATH"
echo "  HF_MODEL=$HF_MODEL"
echo "  MERGE_OUTPUT_DIR=$MERGE_OUTPUT_DIR"
echo ""

torchrun --nproc-per-node=1 \
  /opt/Megatron-Bridge/examples/peft/merge_lora.py \
  --lora-checkpoint "$LORA_CHECKPOINT" \
  --output "$MERGE_OUTPUT_DIR" \
  --hf-model-path "$HF_MODEL" \
  --pretrained "$BASE_MODEL_PATH" \
  --cpu

---
## Step 5: Export to HuggingFace Format

Convert the merged Megatron-Bridge checkpoint to HuggingFace format for inference or uploading to the Hub.

In [ ]:
%%bash
set -euo pipefail

# Output sits next to the merged checkpoint with _hf_format suffix
HF_PATH="${MERGE_OUTPUT_DIR}_hf_format"
mkdir -p "$HF_PATH"

echo "=== Export Environment ==="
echo "  MERGE_OUTPUT_DIR=$MERGE_OUTPUT_DIR"
echo "  HF_MODEL=$HF_MODEL"
echo "  HF_PATH=$HF_PATH"
echo ""

torchrun --nproc-per-node=$N_DEVICES \
  /opt/Megatron-Bridge/examples/conversion/convert_checkpoints.py export \
  --hf-model "$HF_MODEL" \
  --megatron-path "$MERGE_OUTPUT_DIR" \
  --hf-path "$HF_PATH" \
  --tp 1 --ep $N_DEVICES

---
## Step 6: Deploying the Trained Model for Inference

You can deploy the model you just trained by following the cookbook under [`usage-cookbook/Nemotron-3-Super/vllm_cookbook.ipynb`](https://github.com/NVIDIA-NeMo/Nemotron/tree/main/usage-cookbook/Nemotron-3-Super/vllm_cookbook.ipynb) and providing the local path of the merged checkpoint (i.e. the path stored in `$MERGE_OUTPUT_DIR`) as the `model_id`.